# Explore Architectures

Parameter counts and layer-by-layer breakdown for CNN, DualCSN, and DualSSN.
Use this to verify counts against LaTeX tables and compare model complexity.

In [ ]:
import torch
from dcreclass.models import CNN, ScatterNet, DualCNNSqueezeNet, DualScatterSqueezeNet

## Input shapes

Must match what is used in training (`04.train_classifier.py`):
- Image: single-channel 128×128
- Scattering: J=2, L=12, order=2 → 169 coefficients at 32×32

In [ ]:
IMG_SHAPE  = (1, 128, 128)    # (C, H, W)
SCAT_SHAPE = (169, 32, 32)    # J=2, L=12, order=2
J = 2

cnn     = CNN(input_shape=IMG_SHAPE, num_classes=2)
dualcsn = DualCNNSqueezeNet(input_shape=IMG_SHAPE, num_classes=2)
dualssn = DualScatterSqueezeNet(img_shape=IMG_SHAPE, scat_shape=SCAT_SHAPE, num_classes=2, J=J)

print("Models instantiated successfully.")

## Parameter counting utilities

In [ ]:
def count_params(module):
    """Count trainable parameters in a module."""
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


def section_params(model):
    """Print parameter count per named top-level child, and the grand total."""
    for name, module in model.named_children():
        n = count_params(module)
        print(f"  {name:30s}  {n:>10,}")
    print(f"  {'TOTAL':30s}  {count_params(model):>10,}")


def layer_detail(seq, label):
    """Walk a Sequential and print params per sub-layer."""
    print(f"\n  [{label}]")
    total = 0
    for i, layer in enumerate(seq):
        n = count_params(layer)
        if n > 0:
            print(f"    [{i:02d}] {type(layer).__name__:30s}  {n:>8,}")
        total += n
    print(f"    subtotal: {total:,}")
    return total

## Section-level summary (all models)

In [ ]:
print("=" * 55)
print("CNN")
print("=" * 55)
section_params(cnn)

print()
print("=" * 55)
print("DualCNNSqueezeNet (DualCSN)")
print("=" * 55)
section_params(dualcsn)

print()
print("=" * 55)
print(f"DualScatterSqueezeNet (DualSSN)  J={J}")
print("=" * 55)
section_params(dualssn)

## Per-layer detail — DualSSN

In [ ]:
print("--- DualSSN layer detail ---")
t1 = layer_detail(dualssn.cnn_encoder,        "cnn_encoder")
t2 = layer_detail(dualssn.conv_to_latent_img,  "conv_to_latent_img")
t3 = layer_detail(dualssn.conv_to_latent_scat, "conv_to_latent_scat")
clf = (count_params(dualssn.FC_input)
     + count_params(dualssn.bn1)
     + count_params(dualssn.FC_hidden)
     + count_params(dualssn.bn2)
     + count_params(dualssn.FC_classifier))
print(f"\n  [classifier head]  {clf:,}")
print(f"\n  Grand total (manual sum): {t1+t2+t3+clf:,}")

## Per-layer detail — DualCSN

In [ ]:
print("--- DualCSN layer detail ---")
t1 = layer_detail(dualcsn.img_encoder,      "img_encoder")
t2 = layer_detail(dualcsn.img_to_latent,    "img_to_latent")
t3 = layer_detail(dualcsn.detail_to_latent, "detail_to_latent")
clf = (count_params(dualcsn.FC_input)
     + count_params(dualcsn.bn1)
     + count_params(dualcsn.FC_hidden)
     + count_params(dualcsn.bn2)
     + count_params(dualcsn.FC_classifier))
print(f"\n  [classifier head]  {clf:,}")
print(f"\n  Grand total (manual sum): {t1+t2+t3+clf:,}")

## Per-layer detail — CNN

In [ ]:
print("--- CNN layer detail ---")
t1 = layer_detail(cnn.features,   "features")
t2 = layer_detail(cnn.classifier, "classifier")
print(f"\n  Grand total (manual sum): {t1+t2:,}")